<a href="https://colab.research.google.com/github/aniray2908/nlp-llm-journey/blob/main/01_nlp_fundamentals/demos/03_bag_of_words_tfidf.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Cell 1 — Imports**

In [1]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

**Cell 2 — Simple Bag of Words Example**

In [2]:
from sklearn.feature_extraction.text import CountVectorizer

documents = [
    "the cat sat on the mat",
    "the dog played in the park",
    "cats and dogs are friends",
    "the mat was comfortable",
]

# Create vectoriser
vectorizer = CountVectorizer()
bow_matrix = vectorizer.fit_transform(documents)

# Get vocabulary
vocab = vectorizer.get_feature_names_out()
print(f"Vocabulary size: {len(vocab)}")
print(f"Vocabulary: {vocab}")

# Convert to dense for readability
bow_dense = bow_matrix.toarray()
print(f"\nBoW matrix shape: {bow_dense.shape}")
print(f"(documents={bow_dense.shape[0]}, vocab={bow_dense.shape[1]})\n")

# Show as DataFrame
df = pd.DataFrame(bow_dense, columns=vocab)
df['document'] = documents
print(df[['document', 'the', 'cat', 'dog', 'mat', 'park']].to_string(index=False))

Vocabulary size: 16
Vocabulary: ['and' 'are' 'cat' 'cats' 'comfortable' 'dog' 'dogs' 'friends' 'in' 'mat'
 'on' 'park' 'played' 'sat' 'the' 'was']

BoW matrix shape: (4, 16)
(documents=4, vocab=16)

                  document  the  cat  dog  mat  park
    the cat sat on the mat    2    1    0    1     0
the dog played in the park    2    0    1    0     1
 cats and dogs are friends    0    0    0    0     0
   the mat was comfortable    1    0    0    1     0


**Cell 3 — Document Similarity with BoW**

In [3]:
# Compute cosine similarity between all pairs
similarity = cosine_similarity(bow_matrix)

# Create a nice DataFrame
similarity_df = pd.DataFrame(
    similarity,
    index=[f"Doc {i}" for i in range(len(documents))],
    columns=[f"Doc {i}" for i in range(len(documents))]
)

print("Cosine similarity (BoW):")
print(similarity_df.round(3))

# Find most similar pair
np.fill_diagonal(similarity, -1)  # ignore self-similarity
most_similar_idx = np.unravel_index(similarity.argmax(), similarity.shape)
print(f"\nMost similar pair: Doc {most_similar_idx[0]} and Doc {most_similar_idx[1]}")
print(f"Similarity: {similarity[most_similar_idx[0], most_similar_idx[1]]:.3f}")

Cosine similarity (BoW):
       Doc 0  Doc 1  Doc 2  Doc 3
Doc 0   1.00  0.500    0.0  0.530
Doc 1   0.50  1.000    0.0  0.354
Doc 2   0.00  0.000    1.0  0.000
Doc 3   0.53  0.354    0.0  1.000

Most similar pair: Doc 0 and Doc 3
Similarity: 0.530


**Cell 4 — TF-IDF Vectoriser**

In [4]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Create TF-IDF vectoriser
tfidf_vectorizer = TfidfVectorizer()
tfidf_matrix = tfidf_vectorizer.fit_transform(documents)

# Convert to dense
tfidf_dense = tfidf_matrix.toarray()

# Show as DataFrame
tfidf_df = pd.DataFrame(tfidf_dense, columns=tfidf_vectorizer.get_feature_names_out())
tfidf_df['document'] = documents

# Show key words for each document
print("TF-IDF values (showing non-zero entries):\n")
for i, doc in enumerate(documents):
    print(f"Doc {i}: {doc}")
    nonzero = tfidf_dense[i] > 0
    words = tfidf_vectorizer.get_feature_names_out()[nonzero]
    values = tfidf_dense[i][nonzero]
    for w, v in sorted(zip(words, values), key=lambda x: -x[1])[:5]:  # top 5
        print(f"  {w:<12} {v:.3f}")
    print()

TF-IDF values (showing non-zero entries):

Doc 0: the cat sat on the mat
  the          0.557
  cat          0.436
  on           0.436
  sat          0.436
  mat          0.344

Doc 1: the dog played in the park
  the          0.538
  dog          0.421
  in           0.421
  park         0.421
  played       0.421

Doc 2: cats and dogs are friends
  and          0.447
  are          0.447
  cats         0.447
  dogs         0.447
  friends      0.447

Doc 3: the mat was comfortable
  comfortable  0.575
  was          0.575
  mat          0.453
  the          0.367



**Cell 5 — BoW vs TF-IDF Comparison**

In [5]:
# Compare for a single document
doc_idx = 0
bow_values = bow_dense[doc_idx]
tfidf_values = tfidf_dense[doc_idx]

comparison = pd.DataFrame({
    'word': vocab,
    'BoW': bow_values,
    'TF-IDF': tfidf_values
})
comparison = comparison[comparison['BoW'] > 0].sort_values('BoW', ascending=False)

print(f"Document: '{documents[doc_idx]}'\n")
print(comparison.to_string(index=False))

print("\nObservations:")
print("- BoW: each word gets its raw count")
print("- TF-IDF: common words ('the') get lower weight, content words get higher weight")

Document: 'the cat sat on the mat'

word  BoW   TF-IDF
 the    2 0.557077
 cat    1 0.436384
 mat    1 0.344051
  on    1 0.436384
 sat    1 0.436384

Observations:
- BoW: each word gets its raw count
- TF-IDF: common words ('the') get lower weight, content words get higher weight


**Cell 6 — Document Similarity with TF-IDF**

In [6]:
# Compute cosine similarity
tfidf_similarity = cosine_similarity(tfidf_matrix)

tfidf_sim_df = pd.DataFrame(
    tfidf_similarity,
    index=[f"Doc {i}" for i in range(len(documents))],
    columns=[f"Doc {i}" for i in range(len(documents))]
)

print("Cosine similarity (TF-IDF):")
print(tfidf_sim_df.round(3))

# Compare: which approach gives higher similarity?
print("\nComparison:")
print(f"BoW      Doc0-Doc1: {similarity_df.iloc[0, 1]:.3f}")
print(f"TF-IDF   Doc0-Doc1: {tfidf_sim_df.iloc[0, 1]:.3f}")
print("\nWhy different? TF-IDF weights words differently, changing the geometry.")

Cosine similarity (TF-IDF):
       Doc 0  Doc 1  Doc 2  Doc 3
Doc 0   1.00  0.300    0.0  0.360
Doc 1   0.30  1.000    0.0  0.197
Doc 2   0.00  0.000    1.0  0.000
Doc 3   0.36  0.197    0.0  1.000

Comparison:
BoW      Doc0-Doc1: 0.500
TF-IDF   Doc0-Doc1: 0.300

Why different? TF-IDF weights words differently, changing the geometry.


**Cell 7 — Text Classification with BoW**

In [7]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, confusion_matrix

# Training data
train_docs = [
    "I love this movie, it's amazing!",
    "This film is terrible, I hated it.",
    "Great acting and a wonderful plot!",
    "Waste of time, absolutely dreadful.",
    "Best movie I've seen all year!",
    "Don't bother watching this garbage.",
]
train_labels = [1, 0, 1, 0, 1, 0]  # 1=positive, 0=negative

# Vectorise
vectorizer = CountVectorizer()
X_train = vectorizer.fit_transform(train_docs)

# Train classifier
clf = MultinomialNB()
clf.fit(X_train, train_labels)

# Test on training data
predictions = clf.predict(X_train)
accuracy = accuracy_score(train_labels, predictions)

print(f"Training accuracy: {accuracy:.1%}")
print(f"\nConfusion matrix:")
print(confusion_matrix(train_labels, predictions))

# Show most informative features
feature_names = vectorizer.get_feature_names_out()
print(f"\nMost positive words (highest log probability ratio):")
pos_indices = np.argsort(clf.feature_log_prob_[1] - clf.feature_log_prob_[0])[-5:]
for idx in reversed(pos_indices):
    print(f"  {feature_names[idx]}")

print(f"\nMost negative words:")
neg_indices = np.argsort(clf.feature_log_prob_[1] - clf.feature_log_prob_[0])[:5]
for idx in neg_indices:
    print(f"  {feature_names[idx]}")

Training accuracy: 100.0%

Confusion matrix:
[[3 0]
 [0 3]]

Most positive words (highest log probability ratio):
  movie
  year
  ve
  wonderful
  seen

Most negative words:
  absolutely
  don
  bother
  is
  hated


**Cell 8 — Predicting on New Documents**

In [8]:
# New documents to classify
test_docs = [
    "I loved every minute of it!",
    "This movie was a complete disaster.",
    "Amazing cinematography and acting.",
    "Boring and predictable, don't watch.",
]

# Vectorise using the SAME vectoriser
X_test = vectorizer.transform(test_docs)

# Predict
predictions = clf.predict(X_test)
proba = clf.predict_proba(X_test)

results = pd.DataFrame({
    'document': test_docs,
    'prediction': ['positive' if p == 1 else 'negative' for p in predictions],
    'confidence': proba.max(axis=1)
})

print("Predictions on new documents:")
print(results.to_string(index=False))

Predictions on new documents:
                            document prediction  confidence
         I loved every minute of it!   negative    0.666667
 This movie was a complete disaster.   positive    0.666667
  Amazing cinematography and acting.   positive    0.888889
Boring and predictable, don't watch.   negative    0.500000


**Cell 9 — Sparsity Analysis**

In [10]:
from scipy.sparse import csr_matrix

sparsity_bow = (1 - bow_matrix.nnz / (bow_matrix.shape[0] * bow_matrix.shape[1])) * 100
sparsity_tfidf = (1 - tfidf_matrix.nnz / (tfidf_matrix.shape[0] * tfidf_matrix.shape[1])) * 100

print(f"BoW matrix sparsity:")
print(f"  Shape: {bow_matrix.shape}")
print(f"  Non-zero elements: {bow_matrix.nnz}")
print(f"  Sparsity: {sparsity_bow:.1f}%")

print(f"\nTF-IDF matrix sparsity:")
print(f"  Shape: {tfidf_matrix.shape}")
print(f"  Non-zero elements: {tfidf_matrix.nnz}")
print(f"  Sparsity: {sparsity_tfidf:.1f}%")

print(f"\nFor comparison, an embedding matrix would be DENSE:")
print(f"  If we embedded each of {len(vocab)} words as 300-dim vectors,")
print(f"  with {len(documents)} documents at ~5 words each:")
density_bow = 100 - sparsity_bow
print(f"  Dense matrix would have ~100% non-zero values (vs ~{density_bow:.1f}% here)")

BoW matrix sparsity:
  Shape: (4, 16)
  Non-zero elements: 19
  Sparsity: 70.3%

TF-IDF matrix sparsity:
  Shape: (4, 16)
  Non-zero elements: 19
  Sparsity: 70.3%

For comparison, an embedding matrix would be DENSE:
  If we embedded each of 16 words as 300-dim vectors,
  with 4 documents at ~5 words each:
  Dense matrix would have ~100% non-zero values (vs ~29.7% here)


**Cell 10 — Document Clustering Hint**

In [11]:
from sklearn.cluster import KMeans

# Create more documents
docs = [
    "cats are cute and fluffy",
    "dogs love to play fetch",
    "kittens are adorable pets",
    "puppies are loyal and fun",
    "birds can fly very high",
    "parrots talk and repeat words",
]

# Vectorise
vec = TfidfVectorizer()
X = vec.fit_transform(docs)

# Cluster
kmeans = KMeans(n_clusters=2, random_state=42)
labels = kmeans.fit_predict(X)

# Show results
for doc, label in zip(docs, labels):
    print(f"Cluster {label}: {doc}")

print("\nWithout any labels, TF-IDF + KMeans grouped:")
print("- Cluster 0: Documents about cats (cute, fluffy, adorable)")
print("- Cluster 1: Documents about dogs (play, loyal, fun)")
print("\nThis is unsupervised learning — no labels, just similarity.")

Cluster 0: cats are cute and fluffy
Cluster 0: dogs love to play fetch
Cluster 0: kittens are adorable pets
Cluster 0: puppies are loyal and fun
Cluster 1: birds can fly very high
Cluster 0: parrots talk and repeat words

Without any labels, TF-IDF + KMeans grouped:
- Cluster 0: Documents about cats (cute, fluffy, adorable)
- Cluster 1: Documents about dogs (play, loyal, fun)

This is unsupervised learning — no labels, just similarity.
